<a href="https://colab.research.google.com/github/himanshugajbhiyebhai302-hash/DEEPLEARNING/blob/main/MIT_LAB_2_UNBIASING_PRACTICEpynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import IPython
IPython.display.YouTubeVideo("59bMh59JQDo")

In [ ]:
## Comet ML
!pip install comet_ml --quiet
import comet_ml

from google.colab import userdata
COMET_API_KEY = userdata.get('comet_ml')

!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

In [ ]:
import os
import random
import functools
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Import torch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

# cuda language
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cudnn.benchmark = True

In [ ]:
CACHE_DIR = Path.home() / ".cache"/"mitdeeplearning"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

#get the training data: both from celebimage and imagenet
path_to_training_data = CACHE_DIR.joinpath("train_face_h5")

if path_to_training_data.exists():
  print(f"Using cache  trainiinng data :{path_to_training_data}")
else:
    print(f"Downloading training data to :{path_to_training_data}")
    url = "https://www.dropbox.com/s/hlz8atheyozp1yx/train_face.h5?dl=1"
    torch.hub.download_url_to_file(url,path_to_training_data)

# Instantiate a TrainingDatasetLoader using the download dataset
channels_last = False
loader = mdl.lab2.TrainingDatasetLoader(
    path_to_training_data, channels_last=channels_last
 )


In [ ]:
number_of_training_examples = loader.get_train_size()
(images, labels) = loader.get_batch(100)

In [ ]:
B, C, H, W = images.shape

In [ ]:
face_images  = images[np.where(labels == 1)[0]].transpose(0,2,3,1)
not_face_images = images[np.where(labels == 0)[0]].transpose(0,2,3,1)

idx_face = 28    # @param {type:"slider", min:0, max:50, step:1}
idx_not_face = 16  # @param {type:"slider", min:0, max:50, step:1}
idx_face_2 = 44 #@param{type:"slider", min:0,max:50,step:1}
idx_not_face_2 = 11 #@param{type:"slider",min:0,max:50,step:1}

plt.figure(figsize=(5,5))
plt.subplot(1,2,1)
plt.imshow(face_images[idx_face])
plt.title("Face")
plt.grid(False)

plt.subplot(1,2,2)
plt.title("not face")
plt.imshow(not_face_images[idx_not_face])
plt.grid(False)

plt.figure(figsize=(5,5))
plt.subplot(1,2,1)
plt.imshow(face_images[idx_face_2])
plt.title("Another face")
plt.grid(False)

plt.subplot(1,2,2)
plt.imshow(not_face_images[idx_not_face_2])
plt.title("Another not face image")
plt.grid(False)


plt.show()


sequences = 12
in_channels = images.shape[1]
def make_standard_classification(n_outputs):
  """Create a standard CNN Classifier"""

  # yeh hogyi apni process so apanne,
  # class define ki matlab apna model ke kaam ki process batayi
  class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding=0):
      super().__init__()
      self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
      self.relu = nn.ReLU()
      self.bm = nn.BatchNorm2d(out_channels)

    def forward(self,x):
      x = self.conv(x)
      x = self.relu(x)
      x = self.bm(x)
      return x

   # yeh hogya apna model
  model = nn.Sequential(
      ConvBlock(in_channels,sequences,kernel_size=5,stride=2,padding=2),
      ConvBlock(sequences, 2*sequences,kernel_size=5,stride=2,padding=2),
      ConvBlock(2*sequences, 4*sequences,kernel_size=5,stride=2,padding=2),
      ConvBlock(4*sequences, 6*sequences,kernel_size=5,stride=2,padding=2),
      nn.Flatten(), # Corrected from nn.flatten()
      nn.Linear(H // 16*W // 16*6*sequences, 512),
      nn.ReLU(), # Corrected from nn.ReLU
      nn.Linear(512, n_outputs)

  )

  return model.to(device)

  #Final chapter
standard_classifier = make_standard_classification(n_outputs=1)
print(standard_classifier)


In [ ]:
sequences = 12
in_channels = images.shape[1]

def make_standard_classification(n_outputs):
   """Create a standard CNN Classifier"""

   # yeh hogyi apni process so apanne,
   # class define ki matlab apna model ke kaam ki process batayi
   class ConvBlock(nn.Module):
     def __init__(self, in_channels, out_channels, kernel_size, stride, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm2d(out_channels)

     def forward(self,x):
       x = self.conv(x)
       x = self.relu(x)
       x = self.bn(x)
       return x

   #yeh hogya apna model
   model = nn.Sequential(
       ConvBlock(in_channels,sequences,kernel_size=5,stride=2,padding=2),
       ConvBlock(sequences, 2 * sequences,kernel_size=5,stride=2,padding=2),
       ConvBlock(2 * sequences, 4 * sequences,kernel_size=5,stride=2,padding=2),
       ConvBlock(4 * sequences, 6 * sequences,kernel_size=5,stride=2,padding=2),
       nn.Flatten(),
       nn.Linear(H // 16 * W // 16 * 6 * sequences, 512),
       nn.ReLU(),
       nn.Linear(512, n_outputs)
   )

   return model.to(device)

#Final chapter
standard_classifier = make_standard_classification(n_outputs=1)
print(standard_classifier)


In [ ]:
def create_experiment(project_name, params):
    # end any prior experiments
    if "experiment" in locals():
        experiment.end()

    # initiate the comet experiment for tracking
    experiment = comet_ml.Experiment(api_key=COMET_API_KEY, project_name=project_name)
    # log our hyperparameters, defined above, to the experiment
    for param, value in params.items():
        experiment.log_parameter(param, value)
    experiment.flush()

    return experiment

In [ ]:
##Train the standarad model
loss_fn = nn.BCEWithLogitsLoss()
##
params = dict(
    batch_size = 32,
    num_epochs = 2,
    learning_rate  = 5e-4 ,
)

experiment = create_experiment("6S191_Lab2_Part2_CNN", params)

optimizer = optim.Adam(standard_classifier.parameters(), lr=params["learning_rate"])
# define our optimizer
loss_history = mdl.util.LossHistory(smoothing_factor = 0.99)
plotter  = mdl.util.PeriodicPlotter(sec=2, scale="semilogy")
if hasattr(tqdm, "instances"):
  tqdm.instances.clear()

#set the model to train mode
standard_classifier.train()

def train_step(x,y):
  x =torch.from_numpy(x).to(device)
  y = torch.from_numpy(y).to(device)

  # clear the gradiants
  optimizer.zero_grad()

  #feed the images into the model
  logits = standard_classifier(x) # Call the model here
  # compute the loss
  loss = loss_fn(logits, y)

  #Backpropagation
  loss.backward()
  optimizer.step()

  return loss

# The training loop
step = 0
for epoch in range(params["num_epochs"]):
  for idx in tqdm(range(loader.get_train_size() // params["batch_size"])):
    #Grab a batch of training data and propagation through the network
    x, y = loader.get_batch(params["batch_size"]) # Corrected batch_size
    loss = train_step(x,y) # Corrected function name
    loss_value = loss.detach().cpu().numpy()


    loss_history.append(loss_value)
    plotter.plot(loss_history.get())

    experiment.log_metric("loss", loss_value, step=step)
    step += 1

experiment.end()

In [ ]:
# Set the model for evaluation
standard_classifier.eval()

# TRAINING DATA
# Evaluate on a dataset of CeleBa + Imagenet
(batch_x, batch_y) = loader.get_batch(5000)
batch_x = torch.from_numpy(batch_x).float().to(device)
batch_y = torch.from_numpy(batch_y).float().to(device)

with torch.inference_mode():
    y_pred_logits = standard_classifier(batch_x)
    y_pred_standard = torch.round(torch.sigmoid(y_pred_logits))

    #Accuracy
    acc_standard = torch.mean((batch_y == y_pred_standard).float())

print(
    "Standard CNN accuracy on (potentially biased) training set: {:.4f}".format(
        acc_standard.item()
    )
)



In [ ]:
### Load test dataset and plot examples ###

test_faces = mdl.lab2.get_test_faces(channels_last=channels_last)
keys = ["Light Female", "Light Male", "Dark Female", "Dark Male"]

fig, axs = plt.subplots(1,len(keys),
                        figsize=(7.5, 7.5))
for i, (group, key) in enumerate(zip(test_faces, keys)):
  axs[i].imshow(np.hstack(group).transpose(1,2,0))
  axs[i].set_title(key, fontsize=15)
  axs[i].axis("off")

In [ ]:
### Evaluate the standard CNN on the test data ##
standard_classifier_probs_list = []

with torch.inference_mode():
  for x in test_faces:
    x = torch.from_numpy(np.array(x, dtype=np.float32)).to(device)
    logits = standard_classifier(x)
    probs = torch.sigmoid(logits)
    probs = torch.squeeze(probs, dim=-1)
    standard_classifier_probs_list.append(probs.cpu().numpy())

standard_classifier_probs = np.array(standard_classifier_probs_list)

# Plot the prediction accuracies per demograpic
xx = range(len(keys))
yy = standard_classifier_probs.mean(axis=1)
plt.bar(xx, yy)
plt.xticks(xx,keys)
plt.ylim(max(0, yy.min() - np.ptp(yy) / 2.0), yy.max() + np.ptp(yy) / 2.0)
plt.title("Standard classifier predictions")

###Mitigating algorithmic bias
Imbalances in the training data can result in unwanted algorithmic bias. For example, the majority of faces in CelebA (our training set) are those of light-skinned females. As a result, a classifier trained on CelebA will be better suited at recognizing and classifying faces with features similar to these, and will thus be biased.

How could we overcome this? A naive solution -- and one that is being adopted by many companies and organizations -- would be to annotate different subclasses (i.e., light-skinned females, males with hats, etc.) within the training data, and then manually even out the data with respect to these groups.

But this approach has two major disadvantages. First, it requires annotating massive amounts of data, which is not scalable. Second, it requires that we know what potential biases (e.g., race, gender, pose, occlusion, hats, glasses, etc.) to look for in the data. As a result, manual annotation may not capture all the different features that are imbalanced within the training data.

Instead, let's actually learn these features in an unbiased, unsupervised manner, without the need for any annotation, and then train a classifier fairly with respect to these features. In the rest of this lab, we'll do exactly that.



### 2.4 Variational Autoencoders for learning latent
Isme problem yeh hai ki our model can train on a celeb dataset which can lead to dominate light fair skinned people so rare case like blacks and people with hats will gonna to be a problem

Our goal is to train a debiased version of this classifier -- one that accounts for potential disparities in feature(rare cases)representation within the training data. Specifically, to build a debiased facial classifier, we'll train a model that learns a representation of the underlying latent space to the face training data. The model then uses this information to mitigate unwanted biases by sampling faces with rare features, like dark skin or hats, more frequently during training. The key design requirement for our model is that it can learn an encoding of the latent features in the face data in an entirely unsupervised way. To achieve this, we'll turn to variational autoencoders (VAEs).

SO BASICALLY YEH VAE YEH SMJHA THE HAI KI ONLY RELYING ON TRAINING IS NOT SUFFFICIENT APNE KO EK LATENT SPACE BNANA HOGA JISME HUM EK CHEEZ KRSAKTE HAI KI MEAN VECTOR SPACE FROM EACH PHOTO AND STANDARD DEVIATION USKE BASIS PE PTA LAGA SAKTE HAI

The equation for the latent loss is provided by: (encodeing loss)

$$L_{KL}(\mu, \sigma) = \frac{1}{2}\sum_{j=0}^{k-1} (\sigma_j + \mu_j^2 - 1 - \log{\sigma_j})$$

The equation for the reconstruction loss is provided by:(decoding loss)

$$L_{x}{(x,\hat{x})} = ||x-\hat{x}||_1$$

Thus for the VAE loss we have:

$$L_{VAE} = c\cdot L_{KL} + L_{x}{(x,\hat{x})}$$

where $c$ is a weighting coefficient used for regularization. Now we're ready to define our VAE loss function:

In [ ]:
### Defining the VAE loss function
def vae_loss_function(x,x_recon,mu, logsigma, kl_weights=0.0005):

  #latent loss mein 0.5 = 1/2 summantion = torch.mean and log of sigma = logsigma and sigma = logsigma
  #logsigma mein se log nikalna hai toh exponential se multiply kro

    latent_loss = 0.5 * torch.sum(torch.exp(logsigma) + mu**2 - 1.0 - logsigma)
    reconstruction_loss = torch.mean(torch.abs(x - x_recon))
    vae_loss = kl_weights * latent_loss + reconstruction_loss
    return vae_loss


### Understanding VAEs: reparameterization

As you may recall from lecture, VAEs use a "reparameterization  trick" for sampling learned latent variables. Instead of the VAE encoder generating a single vector of real numbers for each latent variable, it generates a vector of means and a vector of standard deviations that are constrained to roughly follow Gaussian distributions. We then sample from the standard deviations and add back the mean to output this as our sampled latent vector. Formalizing this for a latent variable $z$ where we sample $\epsilon \sim N(0,(I))$ we have:

$$z = \mu + e^{\left(\frac{1}{2} \cdot \log{\Sigma}\right)}\circ \epsilon$$

where $\mu$ is the mean and $\Sigma$ is the covariance matrix. This is useful because it will let us neatly define the loss function for the VAE, generate randomly sampled latent variables, achieve improved network generalization, **and** make our complete VAE network differentiable so that it can be trained via backpropagation. Quite powerful!

Let's define a function to implement the VAE sampling operation:

In [ ]:
### VAE reparameterization
#eps = epsilon(random noise sample)
def sampling(z_mean, z_logsigma):
  eps = torch.radn_like(z_mean)

  z = z_mean + torch.exp(0.5 * z_logsigma) * eps
  return z

###2.5 Debiasing variational autoencoder (DB-VAE)
Now, we'll use the general idea behind the VAE architecture to build a model, termed a debiasing variational autoencoder or DB-VAE, to mitigate (potentially) unknown biases present within the training idea. We'll train our DB-VAE model on the facial detection task, run the debiasing operation during training, evaluate on the PPB dataset, and compare its accuracy to our original, biased CNN model.

The DB-VAE model
The key idea behind this debiasing approach is to use the latent variables learned via a VAE to adaptively re-sample the CelebA data during training. Specifically, we will alter the probability that a given image is used during training based on how often its latent features appear in the dataset. So, faces with rarer features (like dark skin, sunglasses, or hats) should become more likely to be sampled during training, while the sampling probability for faces with features that are over-represented in the training dataset should decrease (relative to uniform random sampling across the training data).

A general schematic of the DB-VAE approach is shown here:

Defining the DB-VAE loss function
This means we'll need to be a bit clever about the loss function for the DB-VAE. The form of the loss will depend on whether it's a face image or a non-face image that's being considered.

For face images, our loss function will have two components:

VAE loss ( LVAE ): consists of the latent loss and the reconstruction loss.
Classification loss ( Ly(y,y^) ): standard cross-entropy loss for a binary classification problem.
In contrast, for images of non-faces, our loss function is solely the classification loss.

We can write a single expression for the loss by defining an indicator variable  If which reflects which training data are images of faces ( If(y)=1  ) and which are images of non-faces ( If(y)=0 ). Using this, we obtain:

Ltotal=Ly(y,y^)+If(y)[LVAE]

### BASICALLY YEH JO DEBIASING VAE KA CASE HAI WOH RESAMPLING KA CASE HAI AFTER DECODING OF VECTOR MEAN AND STANDARD DEVIATION

###NOTE:
jisme loss mein ek characterization loss bhi judega

In [ ]:
## Loss function DB -VAE
def debiasing_loss_function(x, x_pred, y , y_logit, mu, logsigma):
  vae_loss = vae_loss_function(x,x_pred, mu, logsigma)
  classification_loss = F .binary_cross_entropy(y_logit,y)
  y = y.float()
  face_indicator = (y == 1.0).float()

  total_loss = classification_loss + face_indicator(vae_loss)